In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("orders_raw.csv")
df.head()

,order_id,customer_id,order_date,category,quantity,unit_price,discount,returned
0,4913,1116,2026-09-17,Tech,2,$115.41,20%,Yes
1,2130,96,2026-07-25,Clothing,4,$148.20,10%,No
2,3816,430,2026-09-12,Home,4,$109.99,20%,No
3,2894,329,2026-08-30,Tech,3,$101.31,10%,No
4,446,24,2026-08-17,Tech,4,$64.66,20%,No


Identify and remove duplicate orders.

In [3]:
df.drop_duplicates(inplace=True)

In [4]:
df.duplicated().sum()

np.int64(0)

Normalize categories and convert currency, percentages, dates, and return flags.

In [5]:
#Convert dates
df["order_date"] = pd.to_datetime(df["order_date"] , format="mixed", errors="coerce")
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6000 entries, 0 to 6079
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     6000 non-null   int64         
 1   customer_id  6000 non-null   int64         
 2   order_date   5980 non-null   datetime64[ns]
 3   category     6000 non-null   object        
 4   quantity     6000 non-null   int64         
 5   unit_price   5900 non-null   object        
 6   discount     6000 non-null   object        
 7   returned     6000 non-null   object        
dtypes: datetime64[ns](1), int64(3), object(4)
memory usage: 421.9+ KB


In [6]:
#Normalize categories
df["category"] = df["category"].str.strip().str.capitalize()
df["category"].unique()

array(['Tech', 'Clothing', 'Home'], dtype=object)

In [7]:
#Convert unit_price & discount
df["unit_price"] = pd.to_numeric(
    df["unit_price"].str.replace(r"[$,]", "", regex=True),
    errors="coerce"
)
df["discount"] = pd.to_numeric(
    df["discount"].str.replace("%", "", regex=False),
    errors="coerce"
) / 100
df

,order_id,customer_id,order_date,category,quantity,unit_price,discount,returned
0,4913,1116,2026-09-17,Tech,2,115.41,0.2,Yes
1,2130,96,2026-07-25,Clothing,4,148.20,0.1,No
2,3816,430,2026-09-12,Home,4,109.99,0.2,No
3,2894,329,2026-08-30,Tech,3,101.31,0.1,No
4,446,24,2026-08-17,Tech,4,64.66,0.2,No
...,...,...,...,...,...,...,...,...
6075,5700,1008,2026-09-07,Tech,3,138.39,0.1,Yes
6076,2551,968,2026-09-30,Tech,4,111.51,0.0,No
6077,538,101,2026-08-06,Clothing,2,74.46,0.0,Yes
6078,1221,392,2026-07-05,Clothing,2,16.03,0.0,No


In [8]:
#Convert return flags
df["returned"] = df["returned"].str.strip().str.lower().map({"yes":True,"no":False})
df

,order_id,customer_id,order_date,category,quantity,unit_price,discount,returned
0,4913,1116,2026-09-17,Tech,2,115.41,0.2,True
1,2130,96,2026-07-25,Clothing,4,148.20,0.1,False
2,3816,430,2026-09-12,Home,4,109.99,0.2,False
3,2894,329,2026-08-30,Tech,3,101.31,0.1,False
4,446,24,2026-08-17,Tech,4,64.66,0.2,False
...,...,...,...,...,...,...,...,...
6075,5700,1008,2026-09-07,Tech,3,138.39,0.1,True
6076,2551,968,2026-09-30,Tech,4,111.51,0.0,False
6077,538,101,2026-08-06,Clothing,2,74.46,0.0,True
6078,1221,392,2026-07-05,Clothing,2,16.03,0.0,False


Decide how to handle missing prices, invalid dates, and nonpositive quantities.

Separate usable rows from quarantined rows, recording rejection reasons.

In [9]:
checks = pd.DataFrame({
    "missing_or_invalid_price": (
        df["unit_price"].isna() | df["unit_price"].lt(0)
    ),
    "invalid_date": df["order_date"].isna(),
    "invalid_quantity": (
        df["quantity"].isna() | df["quantity"].le(0)
    ),
    "invalid_discount": ~df["discount"].between(0, 1),
    "invalid_return_flag": df["returned"].isna(),
    "invalid_category": ~df["category"].isin(
        ["Home", "Tech", "Clothing"]
    ),
}, index=df.index)

reasons = checks.apply(
    lambda row: "; ".join(row.index[row]),
    axis=1
)

rejected = checks.any(axis=1)

orders_quarantine = df.loc[rejected].copy()
orders_quarantine["rejection_reasons"] = reasons.loc[rejected]

orders_clean = df.loc[~rejected].copy()

In [10]:
orders_clean

,order_id,customer_id,order_date,category,quantity,unit_price,discount,returned
0,4913,1116,2026-09-17,Tech,2,115.41,0.2,True
1,2130,96,2026-07-25,Clothing,4,148.20,0.1,False
2,3816,430,2026-09-12,Home,4,109.99,0.2,False
3,2894,329,2026-08-30,Tech,3,101.31,0.1,False
4,446,24,2026-08-17,Tech,4,64.66,0.2,False
...,...,...,...,...,...,...,...,...
6075,5700,1008,2026-09-07,Tech,3,138.39,0.1,True
6076,2551,968,2026-09-30,Tech,4,111.51,0.0,False
6077,538,101,2026-08-06,Clothing,2,74.46,0.0,True
6078,1221,392,2026-07-05,Clothing,2,16.03,0.0,False


In [11]:
orders_clean.isnull().sum()

order_id       0
customer_id    0
order_date     0
category       0
quantity       0
unit_price     0
discount       0
returned       0
dtype: int64

Calculate net revenue and save orders_clean.csv.

In [12]:
orders_clean["net_revenue"] = (
    orders_clean["quantity"]
    * orders_clean["unit_price"]
    * (1 - orders_clean["discount"])
).where(~orders_clean["returned"], 0).round(2)

orders_clean.to_csv("orders_clean.csv", index=False)

orders_quarantine.to_csv(
    "orders_quarantine.csv", index=False
)

In [13]:
print({
    "deduplicated_rows": len(df),
    "usable_rows": len(orders_clean),
    "quarantined_rows": len(orders_quarantine),
})

print("\nRows affected by each issue:")
print(checks.sum())

assert len(df) == len(orders_clean) + len(orders_quarantine)
assert orders_clean["order_id"].is_unique
assert orders_clean["net_revenue"].notna().all()
assert orders_clean.loc[
    orders_clean["returned"], "net_revenue"
].eq(0).all()

{'deduplicated_rows': 6000, 'usable_rows': 5855, 'quarantined_rows': 145}

Rows affected by each issue:
missing_or_invalid_price    100
invalid_date                 20
invalid_quantity             25
invalid_discount              0
invalid_return_flag           0
invalid_category              0
dtype: int64


In [14]:
orders_quarantine

,order_id,customer_id,order_date,category,quantity,unit_price,discount,returned,rejection_reasons
45,2367,763,2026-09-18,Tech,3,NaN,0.1,False,missing_or_invalid_price
48,758,783,2026-07-10,Home,3,NaN,0.1,False,missing_or_invalid_price
49,4101,761,2026-08-15,Clothing,4,NaN,0.0,False,missing_or_invalid_price
108,1219,802,2026-09-10,Tech,1,NaN,0.2,False,missing_or_invalid_price
129,5024,1144,2026-09-28,Tech,2,NaN,0.1,True,missing_or_invalid_price
...,...,...,...,...,...,...,...,...,...
5904,4572,858,NaT,Clothing,2,145.82,0.0,False,invalid_date
5965,3794,843,2026-09-20,Home,-1,52.45,0.2,False,invalid_quantity
6025,93,427,2026-09-13,Clothing,1,NaN,0.1,False,missing_or_invalid_price
6045,4387,656,2026-09-09,Clothing,4,NaN,0.0,False,missing_or_invalid_price
